<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [ ]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan rekayasa fitur

In [ ]:
# Memilih dataset dan parameter pemisahan
import pandas as pd
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data, clean_data, add_aggregate_features, split_data
from pci_ml.memory import configure, release
from pci_ml.ui import build_panel, format_adaptive_df

EXPORT_MODE = True   # True = print panel values instead of showing widgets
RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

df = None
handle = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_adaptive_df(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select, export=EXPORT_MODE)

handle = display(Markdown(""), display_id=True)

on_select(panel.values()["file"])


In [ ]:
# Membersihkan data, menghitung fitur density, dan menambahkan fitur agregat
cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))
df, features = add_aggregate_features(df, features)

if not EXPORT_MODE:
    print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
          f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
          f"PCI minimum {cfg['pci_min']}")

markdown_table = format_adaptive_df(df)
print(f"Data telah dibersihkan; {len(features)} fitur (density + agregat) telah dihitung.")
display(Markdown(markdown_table))


#### 2. Pemisahan data latih dan data uji

In [ ]:
# Memisah data sesuai kriteria
from pci_ml.ui import split_progress

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))
    df, features = add_aggregate_features(df, features)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=split_progress)
print()

split_summary = result["summary"]

# Print the summary outside the progress line so it always shows up
print(split_summary)

release("result", "df")


#### 3. Pelatihan model *machine learning*

In [ ]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import (AGGREGATE_FEATURES, METADATA, PCI, ROAD_ID,
                         latest_checkpoint, load_checkpoint)

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

train_df, test_df, features, meta = load_checkpoint(filename, ckpt_tag)
missing = [name for name in AGGREGATE_FEATURES if name not in features]
if missing:
    raise ValueError(
        "Checkpoint belum memuat fitur agregat. Jalankan ulang Cell 7. "
        f"Kolom hilang: {missing}"
    )

X_train = train_df[features].to_numpy(dtype="float32")
X_test = test_df[features].to_numpy(dtype="float32")
y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
y_test = test_df[PCI].to_numpy(dtype="float32")

groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)

assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

del train_df, test_df

n_tr, n_te = len(X_train), len(X_test)
total = n_tr + n_te
split_text = (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
              f"**Test:** {X_test.shape} ({n_te / total:.2%})")

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_text}"))

release("df", "ckpt_tag", "current")


In [ ]:
# Tuning hyperparameter Random Forest dengan GroupKFold
from IPython.display import Markdown, display, HTML
from pci_ml.random_forest import tune_random_forest, evaluate_random_forest
from pci_ml.ui import build_hyperparameter_table, build_param_panel

TUNING_DEFAULTS = {
    "n_iter": 100,
    "cv_splits": 5,
}

TUNING_SPECS = {
    "n_iter":    {"type": "int", "description": "Jumlah kandidat:", "label": "Jumlah kandidat"},
    "cv_splits": {"type": "int", "description": "Jumlah fold:", "label": "Jumlah fold"},
}

RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}

tuning_panel = build_param_panel(
    TUNING_DEFAULTS, TUNING_SPECS, "Parameter Tuning Random Forest:", export=EXPORT_MODE
)
tuning_params = tuning_panel.values()

display(Markdown(
    f"**Memulai tuning:** {tuning_params['n_iter']} kandidat | "
    f"{tuning_params['cv_splits']} fold CV"
))
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=tuning_params["n_iter"],
    cv_splits=tuning_params["cv_splits"],
    random_state=SEED,
)

rf_best_params = rf_result["params"]
rf_cv = rf_result["cv_pooled"]
tuned_model = rf_result["model"]
tuned_test = evaluate_random_forest(tuned_model, X_test, y_test)

param_explanations = {
    "n_estimators": "Jumlah pohon keputusan",
    "max_depth": "Kedalaman maksimum tiap pohon.",
    "min_samples_split": "Jumlah sampel minimum agar sebuah node boleh dipecah.",
    "min_samples_leaf": "Jumlah sampel minimum di tiap daun.",
    "max_features": "Jumlah fraksi fitur di tiap pohon.",
    "max_samples": "Jumlah data latih yang dipakai untuk tiap pohon.",
}

param_table = build_hyperparameter_table(
    rf_best_params, RF_SEARCH_SPACE.keys(), param_explanations
)

metric_table = (
    "| Evaluasi | MAE | RMSE | R\u00b2 |\n"
    "|---|---:|---:|---:|\n"
    f"| CV pooled OOF | {rf_cv['mae']:.4f} | {rf_cv['rmse']:.4f} | {rf_cv['r2']:.4f} |\n"
    f"| Test | {tuned_test['mae']:.4f} | {tuned_test['rmse']:.4f} | {tuned_test['r2']:.4f} |\n"
)

n_test_roads = test_ids["ROAD_ID"].nunique()
explanation = f"""
**Keterangan**
- **MAE**: rata-rata selisih absolut antara prediksi dan PCI aktual, dalam satuan poin PCI. Semakin kecil semakin baik.
- **RMSE**: akar rata-rata kuadrat error, dalam satuan poin PCI. Lebih sensitif terhadap error besar; semakin kecil semakin baik.
- **R\u00b2**: proporsi variasi PCI yang dijelaskan model. Semakin mendekati 1 semakin baik.
- **CV pooled OOF**: semua prediksi out-of-fold dari {tuning_params['cv_splits']} fold digabung, lalu dihitung sekali. Ini estimasi generalisasi yang lebih stabil daripada rata-rata R\u00b2 per fold, tetapi sedikit optimistis karena hyperparameter dipilih dari CV yang sama.
- **Test**: evaluasi pada data uji yang tidak dipakai saat tuning. Ini estimasi akhir pada {n_test_roads} road holdout, tetapi hanya satu holdout sehingga variansnya lebih tinggi.
"""

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown(f"### Hasil evaluasi\n\n{metric_table}\n\n{explanation}"))


#### 4. Perbaikan model Random Forest

Fitur agregat (`TOTAL_DENSITY` dan `SEVERITY_WEIGHTED`) sudah ditambahkan pada tahap rekayasa fitur di Bagian 1. Bagian ini menguji dua perbaikan tambahan pada lipatan yang sama: bobot sampel dan kalibrasi prediksi. Baseline 55 fitur density tetap dilaporkan sebagai pembanding.

1. **Phase 0** - harness evaluasi dan baseline 55 fitur.
2. **Phase 1** - verifikasi fitur agregat hasil rekayasa fitur.
3. **Phase 2** - bobot sampel untuk kondisi buruk.
4. **Phase 3** - kalibrasi prediksi *post-hoc*.
5. **Phase 4** - perbandingan, uji ketahanan, dan rekomendasi.

Pemilihan konfigurasi hanya memakai CV (*out-of-fold*), bukan data uji.

##### 4.1 Phase 0 - Harness evaluasi dan baseline

Semua eksperimen memakai `FOLDS` yang sama, dihitung sekali dari `groups_train` dengan `GroupKFold(5)`. Baseline memakai 55 fitur density asli (kolom sebelum fitur agregat) agar perbandingan dengan diagnostik sebelumnya tetap konsisten. Parameter RF dikunci ke konfigurasi baseline agar efek tiap perbaikan terisolasi; hasil tuning terbaru tetap ditampilkan di Bagian 3. Fitur agregat yang kini menjadi bagian input diuji pada Phase 1.

In [ ]:
# Phase 0 - harness evaluasi dan baseline 55 fitur
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from pci_ml.data import AGGREGATE_FEATURES
from pci_ml.experiments import evaluate_oof, fit_full, make_folds, oof_predict
from pci_ml.memory import release
from pci_ml.random_forest import rf_estimator

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

RF_BASE_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.2,
    "max_samples": 0.9,
}

# GroupKFold(shuffle=True) dipakai pada Phase 4 dan baru tersedia di sklearn >= 1.6.
if tuple(int(part) for part in sklearn.__version__.split(".")[:2]) < (1, 6):
    raise RuntimeError(
        "Butuh scikit-learn >= 1.6 untuk GroupKFold(shuffle=True); "
        f"versi terpasang {sklearn.__version__}."
    )

FOLDS = make_folds(groups_train, n_splits=5)

# Baseline memakai 55 fitur density; fitur agregat diuji pada Phase 1.
BASE_FEATURES = [name for name in features if name not in AGGREGATE_FEATURES]
base_index = [features.index(name) for name in BASE_FEATURES]
X_TRAIN_BASE = X_train[:, base_index]
X_TEST_BASE = X_test[:, base_index]

base_model = fit_full(rf_estimator(RF_BASE_PARAMS, SEED), X_TRAIN_BASE, y_train)
BASELINE_OOF = oof_predict(
    rf_estimator(RF_BASE_PARAMS, SEED), X_TRAIN_BASE, y_train, FOLDS
)
baseline_eval = evaluate_oof(y_train, BASELINE_OOF, groups_train, FOLDS)

depths = np.array([est.get_depth() for est in base_model.estimators_])
train_mae = mean_absolute_error(y_train, base_model.predict(X_TRAIN_BASE))
first_bin = baseline_eval["frame"]
first_bin = first_bin[(first_bin["pci"] > 0) & (first_bin["pci"] <= 25)]

print("Phase 0 - baseline 55 fitur")
print(f"- scikit-learn : {sklearn.__version__}")
print(f"- pooled OOF : MAE {baseline_eval['pooled']['mae']:.4f} | "
      f"RMSE {baseline_eval['pooled']['rmse']:.4f} | R2 {baseline_eval['pooled']['r2']:.4f}")
print(f"- per fold   : " + " | ".join(f"{value:.2f}" for value in baseline_eval["per_fold"]))
print(f"- train MAE  : {train_mae:.4f} (gap {baseline_eval['pooled']['mae'] - train_mae:.2f})")
print(f"- tree depth : max {depths.max()} | mean {depths.mean():.2f}")
print(f"- PCI <= 40  : n {baseline_eval['low']['n']} | MAE {baseline_eval['low']['mae']:.2f} | "
      f"bias {baseline_eval['low']['bias']:+.2f}")
print(f"- PCI > 85   : n {baseline_eval['high']['n']} | MAE {baseline_eval['high']['mae']:.2f}")
print(f"- bin (0,25] tanpa PCI=0: n {len(first_bin)} | MAE {first_bin['err'].mean():.2f} | "
      f"bias {first_bin['signed'].mean():+.2f}")

print("\nError per rentang PCI (baseline):")
display(baseline_eval["per_bin"].round(3))

baseline_eval.pop("frame", None)
release("base_model", "first_bin", "depths")


**Kesimpulan Phase 0.** Harness mereproduksi baseline 55 fitur: pooled OOF MAE 7.1919, MAE per fold 5.92 / 11.18 / 5.34 / 7.36 / 6.09, dan bias positif besar pada PCI rendah (+31.3 pada bin pertama bila PCI = 0 disertakan; +24.3 bila tidak). Pohon tumbuh penuh (kedalaman maksimum 52, rata-rata 43.8), sehingga regularisasi bukan kunci. Fitur agregat yang ditambahkan di Bagian 1 diuji pada Phase 1.

##### 4.2 Phase 1 - Verifikasi fitur agregat

`TOTAL_DENSITY` (jumlah 55 fitur density) dan `SEVERITY_WEIGHTED` (L=1, M=2, H=3) sudah dihitung di cell 5 dan tersimpan di checkpoint. Phase ini memakai `X_train` yang sudah memuat keduanya, sehingga tidak ada agregasi ulang.

In [ ]:
# Phase 1 - verifikasi fitur agregat hasil rekayasa fitur
import pandas as pd
from pci_ml.data import AGGREGATE_FEATURES
from pci_ml.experiments import evaluate_oof, fit_full, oof_predict
from pci_ml.memory import release
from pci_ml.random_forest import rf_estimator

X_TRAIN_AUG = X_train
X_TEST_AUG = X_test

p1_oof = oof_predict(rf_estimator(RF_BASE_PARAMS, SEED), X_TRAIN_AUG, y_train, FOLDS)
p1_eval = evaluate_oof(y_train, p1_oof, groups_train, FOLDS)

print("Phase 1 - fitur agregat")
print(f"- pooled OOF : MAE {p1_eval['pooled']['mae']:.4f} | "
      f"RMSE {p1_eval['pooled']['rmse']:.4f} | R2 {p1_eval['pooled']['r2']:.4f}")
print(f"- per fold   : " + " | ".join(f"{value:.2f}" for value in p1_eval["per_fold"]))
print(f"- PCI <= 40  : MAE {p1_eval['low']['mae']:.2f} | bias {p1_eval['low']['bias']:+.2f}")
print(f"- PCI > 85   : MAE {p1_eval['high']['mae']:.2f}")

p1_model = fit_full(rf_estimator(RF_BASE_PARAMS, SEED), X_TRAIN_AUG, y_train)
importance = pd.Series(p1_model.feature_importances_, index=features)
print("\nImportance fitur agregat:")
print(importance[AGGREGATE_FEATURES].round(4).to_string())
print("\nTop 10 fitur:")
print(importance.sort_values(ascending=False).head(10).round(4).to_string())

p1_eval.pop("frame", None)
release("p1_model", "importance")


**Kesimpulan Phase 1.** Fitur agregat yang kini dihitung di cell 5 menurunkan pooled OOF MAE dari 7.1919 ke 5.9208 dan memperkecil bias PCI rendah (MAE PCI <= 40 dari 20.55 ke 13.63; bias dari +20.01 ke +13.43). Namun MAE PCI > 85 naik dari 2.52 ke 3.22 (+0.70), sedikit di atas batas 0.5. `SEVERITY_WEIGHTED` dan `TOTAL_DENSITY` menjadi dua fitur terpenting (importance 0.335 dan 0.216, total 0.55). Re-tune tidak diulang karena selisihnya hanya noise dan hyperparameter baseline dipertahankan.

##### 4.3 Phase 2 - Bobot sampel untuk kondisi buruk

Sampel diberi bobot `w = 1 + alpha * (100 - PCI) / 100`, sehingga segmen dengan PCI rendah lebih berpengaruh saat latihan. Metrik evaluasi tetap tanpa bobot. `alpha = 0` berarti baseline 55 fitur.

In [ ]:
# Phase 2 - bobot sampel
import pandas as pd
from IPython.display import display
from pci_ml.experiments import condition_weights, evaluate_oof, oof_predict
from pci_ml.random_forest import rf_estimator

PHASE2_ALPHAS = (0, 1, 2, 4, 6)
phase2_rows = []

for alpha in PHASE2_ALPHAS:
    weights = None if alpha == 0 else condition_weights(y_train, alpha)
    oof = oof_predict(
        rf_estimator(RF_BASE_PARAMS, SEED), X_TRAIN_BASE, y_train, FOLDS,
        sample_weight=weights,
    )
    ev = evaluate_oof(y_train, oof, groups_train, FOLDS)
    phase2_rows.append({
        "alpha": alpha,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "Fold 1": ev["per_fold"][0],
        "Fold 2": ev["per_fold"][1],
        "Fold 3": ev["per_fold"][2],
        "Fold 4": ev["per_fold"][3],
        "Fold 5": ev["per_fold"][4],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
    })
    ev.pop("frame", None)

phase2_table = pd.DataFrame(phase2_rows).set_index("alpha")
print("Phase 2 - bobot sampel (55 fitur density)")
display(phase2_table.round(3))


**Kesimpulan Phase 2.** Bobot menurunkan pooled OOF MAE secara konsisten, dari 7.1919 (`alpha=0`) ke 6.7262 (`alpha=4`), dan memperbaiki MAE PCI <= 40 dari 20.55 ke 18.65 serta biasnya dari +20.01 ke +17.20. `alpha=6` hanya memberi perbaikan marginal (6.6803). MAE PCI > 85 justru membaik (2.52 ke 2.32). Efeknya lebih kecil daripada fitur agregat, tetapi arahnya benar dan tidak merugikan jalan baik.

##### 4.4 Phase 3 - Kalibrasi prediksi *post-hoc*

Kalibrator dipelajari dari prediksi *out-of-fold* data latih saja, lalu diterapkan ke lipatan yang ditahan (*leave-one-fold-out*). Dua metode diuji: regresi linear dan isotonic. Untuk model final, kalibrator dilatih pada seluruh OOF dan diterapkan ke data uji; data uji tidak pernah dipakai untuk melatih kalibrator.

In [ ]:
# Phase 3 - kalibrasi post-hoc
import pandas as pd
from IPython.display import display
from pci_ml.experiments import calibrate_oof, evaluate_oof

phase3_rows = []

for method in ("linear", "isotonic"):
    calibrated = calibrate_oof(BASELINE_OOF, y_train, FOLDS, method)
    ev = evaluate_oof(y_train, calibrated, groups_train, FOLDS)
    phase3_rows.append({
        "method": method,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
    })
    ev.pop("frame", None)

phase3_table = pd.DataFrame(phase3_rows).set_index("method")
print("Phase 3 - kalibrasi pada baseline 55 fitur")
display(phase3_table.round(3))


**Kesimpulan Phase 3.** Kalibrasi isotonic memperbaiki R2 (0.7545 ke 0.7679) dan menurunkan bias PCI rendah (MAE PCI <= 40 dari 20.55 ke 15.04; bias dari +20.01 ke +12.76), tetapi pooled OOF MAE justru naik ke 7.5129 dan MAE PCI > 85 naik ke 2.60. Kalibrasi linear mirip (MAE 7.4376, PCI > 85 3.67). Kalibrasi saja tidak cukup: ia memperbaiki arah bias tetapi tidak menambah informasi fitur. Karena itu kalibrasi diuji sebagai pelengkap fitur agregat.

##### 4.5 Phase 4 - Perbandingan, uji ketahanan, dan rekomendasi

Kandidat dipilih hanya dari CV. Data uji dilaporkan untuk finalis sebagai gambaran akhir, bukan untuk memilih konfigurasi. Uji ketahanan mengulang CV dengan lima pengacakan jalan ke lipatan (`GroupKFold(shuffle=True)`), lalu melaporkan rata-rata dan simpangan baku pooled MAE.

In [ ]:
# Phase 4 - perbandingan, uji ketahanan, dan evaluasi test
import pandas as pd
from IPython.display import display
from pci_ml.experiments import (condition_weights, evaluate_variant,
                                shuffled_cv_mae)
from pci_ml.memory import release
from pci_ml.random_forest import rf_estimator


RF_ESTIMATOR = rf_estimator(RF_BASE_PARAMS, SEED)

variants = {
    "Baseline 55 fitur": evaluate_variant(
        RF_ESTIMATOR, X_TRAIN_BASE, y_train, groups_train, FOLDS,
        X_TEST_BASE, y_test,
    ),
    "Fitur agregat": evaluate_variant(
        RF_ESTIMATOR, X_TRAIN_AUG, y_train, groups_train, FOLDS,
        X_TEST_AUG, y_test,
    ),
    "Bobot a=4": evaluate_variant(
        RF_ESTIMATOR, X_TRAIN_BASE, y_train, groups_train, FOLDS,
        X_TEST_BASE, y_test, sample_weight=condition_weights(y_train, 4),
    ),
    "Kalibrasi isotonic": evaluate_variant(
        RF_ESTIMATOR, X_TRAIN_BASE, y_train, groups_train, FOLDS,
        calibrate="isotonic",
    ),
    "Kombinasi (agregat + bobot a=4 + isotonic)": evaluate_variant(
        RF_ESTIMATOR, X_TRAIN_AUG, y_train, groups_train, FOLDS,
        X_TEST_AUG, y_test, sample_weight=condition_weights(y_train, 4),
        calibrate="isotonic",
    ),
}

rows = []
for name, ev in variants.items():
    rows.append({
        "Kandidat": name,
        "OOF MAE": ev["pooled"]["mae"],
        "OOF RMSE": ev["pooled"]["rmse"],
        "OOF R2": ev["pooled"]["r2"],
        "Fold 1": ev["per_fold"][0],
        "Fold 2": ev["per_fold"][1],
        "Fold 3": ev["per_fold"][2],
        "Fold 4": ev["per_fold"][3],
        "Fold 5": ev["per_fold"][4],
        "PCI<=40 MAE": ev["low"]["mae"],
        "PCI<=40 bias": ev["low"]["bias"],
        "PCI>85 MAE": ev["high"]["mae"],
        "Test MAE": ev["test"]["mae"],
        "Test RMSE": ev["test"]["rmse"],
        "Test R2": ev["test"]["r2"],
    })
comparison = pd.DataFrame(rows).set_index("Kandidat")
print("Phase 4 - perbandingan kandidat")
display(comparison.round(3))

# Uji ketahanan: 5 pengacakan jalan ke lipatan
robust_rows = []
for name, X_matrix, weights, calibrate in (
    ("Baseline 55 fitur", X_TRAIN_BASE, None, None),
    ("Fitur agregat", X_TRAIN_AUG, None, None),
    ("Bobot a=4", X_TRAIN_BASE, condition_weights(y_train, 4), None),
    ("Kombinasi", X_TRAIN_AUG, condition_weights(y_train, 4), "isotonic"),
):
    values = shuffled_cv_mae(
        RF_ESTIMATOR, X_matrix, y_train, groups_train,
        sample_weight=weights, calibrate=calibrate,
    )
    robust_rows.append({
        "Kandidat": name,
        "MAE mean": values.mean(),
        "MAE std": values.std(),
        "MAE min": values.min(),
        "MAE max": values.max(),
    })
robustness = pd.DataFrame(robust_rows).set_index("Kandidat")
print("\nUji ketahanan (5 pengacakan jalan ke lipatan):")
display(robustness.round(4))

release("variants", "robust_rows", "rows")


**Kesimpulan Phase 4 dan rekomendasi.**

| Perbaikan | OOF MAE | Bias PCI <= 40 | MAE PCI > 85 | Test MAE |
|---|---:|---:|---:|---:|
| Baseline 55 fitur | 7.1919 | +20.01 | 2.52 | 5.753 |
| Fitur agregat | 5.9208 | +13.43 | 3.22 | 4.768 |
| Bobot a=4 | 6.7262 | +17.20 | 2.32 | 5.510 |
| Kalibrasi isotonic | 7.5129 | +12.76 | 2.60 | - |
| Kombinasi | 5.9242 | +7.83 | 2.97 | 4.868 |

- **Fitur agregat** adalah perbaikan terbesar: pooled OOF MAE 7.19 ke 5.92. Biayanya MAE PCI > 85 naik 0.70.
- **Bobot sampel `alpha=4`** menurunkan MAE ke 6.73 dan memperbaiki bias PCI rendah, tetapi MAE PCI > 85 tetap baik (2.32). Efeknya lebih kecil daripada fitur agregat.
- **Kalibrasi isotonic** sendirian tidak memperbaiki MAE, tetapi sebagai pelengkap kombinasi ia menurunkan MAE PCI > 85 ke 2.97 (+0.45, masih dalam batas) dan bias PCI <= 40 ke +7.83.
- **Kombinasi terbaik** yang memenuhi seluruh kriteria: fitur agregat + bobot `alpha=4` + kalibrasi isotonic. Pooled OOF MAE 5.9242, 4 dari 5 fold membaik (hanya fold 4 naik 7.36 ke 7.49), bias PCI rendah menyusut jelas, dan MAE PCI > 85 hanya naik 0.45.
- **Uji ketahanan** (5 pengacakan jalan): baseline 7.097 +/- 0.104, fitur agregat 6.016 +/- 0.055, bobot a=4 6.624 +/- 0.061, kombinasi 6.033 +/- 0.127. Kombinasi dan fitur agregat praktis setara; kombinasi menukar sedikit MAE rata-rata dengan bias PCI rendah yang jauh lebih baik.
- Hyperparameter baseline dipertahankan karena re-tune sebelumnya hanya memberi selisih noise.

**Keterbatasan.** Hanya 27 jalan latih dan 7 jalan uji. Variasi antar-pengacakan lipatan sekitar +/-0.06-0.13 MAE, sehingga selisih di bawah ~0.2 dianggap noise. Data uji kemungkinan lebih mudah daripada CV, jadi angka uji cenderung optimistis.

### *Source code* modul dan fungsi

- `src/pci_ml/data.py` - pembersihan data, rekayasa fitur agregat, split, dan checkpoint.
- `src/pci_ml/random_forest.py` - training, tuning, dan evaluasi Random Forest.
- `src/pci_ml/experiments.py` - fold, OOF, metrik, bobot, dan kalibrasi yang model-agnostic.
- `src/pci_ml/ui.py` - panel, progress split, dan tabel hyperparameter.
- `src/pci_ml/memory.py` - manajemen memori notebook.